# Ma Soi — Behavior Cloning locally (P0)

Train BC with P0 flags from the encoded dataset in `.tmp/enc` (§17).
The Colab version is `train_bc_colab.ipynb` — this one is local, no Drive/GPU.

## Before running

- Dataset: `.tmp/enc` (checked in section 1). If missing:
  `npm run ai:encode -- --in .tmp/bc/trajectories.jsonl --out .tmp/enc`.
- Any kernel works: torch/matplotlib run via the venv as subprocesses,
  the check cell only needs numpy (system python has it).
- CPU-only: several times slower than a Colab T4. Small net (96K params) —
  40 epochs still finish within a session, just let it run.
- Boundary (§39) unchanged: numbers from `.bin` only, no trajectory parsing.

## 0. Environment

Locate the repo root (open the notebook from the repo root or anywhere below it),
the venv and the dataset. If this fails, nothing below can run.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

_here = Path.cwd()
# Walk up from the notebook dir (VS Code sets cwd to ai-training/colab/)
# until something looks like the repo root.
ROOT = None
for _p in [_here, *_here.parents]:
    if (_p / 'ai-training' / 'masoi_training').exists():
        ROOT = _p
        break
if ROOT is None:
    raise SystemExit(f'cannot find repo root from {Path.cwd()} - open the notebook from the repo root')
VENV_PY = ROOT / 'ai-training' / '.venv' / 'Scripts' / 'python.exe'
TRAIN_DIR = ROOT / 'ai-training'
# Subprocesses inherit the kernel environment - notebook kernels set
# MPLBACKEND=module://matplotlib_inline... which the venv matplotlib lacks.
# Pin it here: every venv call uses the Agg backend (file output, no display).
VENV_ENV = {**os.environ, 'MPLBACKEND': 'Agg'}
DATA = ROOT / '.tmp' / 'enc-m'
SMOKE = ROOT / '.tmp' / 'smoke-local'
OUT = ROOT / '.tmp' / 'model-m'
print('ROOT ', ROOT)
print('DATA ', DATA, '->', 'found' if (DATA / 'meta.json').exists() else 'MISSING')
print('venv ', VENV_PY, '->', 'found' if VENV_PY.exists() else 'MISSING')
if not VENV_PY.exists():
    raise SystemExit('missing venv - install per ai-training/README.md first')
if not (DATA / 'meta.json').exists():
    raise SystemExit('missing .tmp/enc - run ai:encode first')

# torch/matplotlib live in the venv only: ask via subprocess so any kernel works.
ver = subprocess.run(
    [str(VENV_PY), '-c',
     'import torch; print(torch.__version__, torch.cuda.is_available(), torch.get_num_threads())'],
    capture_output=True, text=True, cwd=str(TRAIN_DIR), env=VENV_ENV,
)
print('torch in venv (version, cuda, threads):', (ver.stdout or ver.stderr).strip())
mpl = subprocess.run(
    [str(VENV_PY), '-c', 'import matplotlib; print(matplotlib.__version__)'],
    capture_output=True, text=True, env=VENV_ENV,
)
print('matplotlib (venv):', (mpl.stdout or mpl.stderr).strip())
print('this kernel:', sys.executable)

## 1. Validate data before training

Three questions before spending a CPU session:

1. Do files match `meta.json`?
2. Do train/val/test sizes sum to the full set (§15)?
3. **Is every label LEGAL under its own mask**?

Same as the Colab version: check inside a function, return numbers not arrays,
so RAM is released before the train cell spawns its subprocess (~6 GB peak full package).

In [ ]:
import gc

sys.path.insert(0, str(TRAIN_DIR))
from masoi_training.data import action_distribution, load


def check(path):
    data = load(str(path))  # throws if a file mismatches meta.json

    sizes = {name: len(data.split(name)) for name in ('train', 'validation', 'test')}
    assert sum(sizes.values()) == len(data)
    assert data.masks[range(len(data)), data.actions].all(), 'labels point at ILLEGAL actions'
    assert set(data.rewards.tolist()) <= {-1.0, 1.0}
    assert data.optimal is not None, 'missing optimal.u8.bin - re-run ai:encode'

    return {
        'version': data.meta.get('datasetVersion'),
        'commit': (data.meta.get('gitCommit') or '')[:8],
        'rows': len(data),
        'obs': data.obs_size,
        'actions': data.action_size,
        'games': data.meta.get('games'),
        'sizes': sizes,
        'classes': len(action_distribution(data)),
        'scores': data.scores is not None,
    }


info = check(DATA)
gc.collect()

print('dataset  ', info['version'], 'commit', info['commit'])
print('rows     ', info['rows'], '| obs', info['obs'], '| actions', info['actions'])
print('games    ', info['games'])
print('split    ', info['sizes'])
print('action classes (§43):', info['classes'])
print('scores   ', 'present (only used with --distill-alpha > 0)' if info['scores'] else 'absent - not needed by default')
print('\nOK - ready to train. RAM was released before the train cell.')

## 2. Smoke test: 2 epochs

Catch config errors here in minutes instead of after a full CPU run.

In [ ]:
import shutil
import subprocess

shutil.rmtree(SMOKE, ignore_errors=True)
done = subprocess.run(
    [str(VENV_PY), '-m', 'masoi_training.train_bc',
     '--data', str(DATA), '--out', str(SMOKE), '--epochs', '2'],
    cwd=str(TRAIN_DIR), env=VENV_ENV,
)
if done.returncode != 0:
    raise SystemExit(f'smoke test failed (exit {done.returncode}) - read the log above')
print('smoke OK')

## 3. Full train (P0)

P0 flags on: `--optimizer adamw` + `--weight-decay 0.01` + `--scheduler cosine`
against overfit, `--grad-clip 1.0` against exploding steps, `--patience 5` to stop
early keeping the best epoch, `--init orthogonal` for a stable start. All recorded
in `trainingConfig` inside `metrics.json`.

`--batch-size 512` matches every other report in the repo — keep it comparable.

P1-1 opt-in (commented in CMD): `--activation silu` / `--norm layernorm` export
`masoi-mlp-2`, which the engine now runs. Try only after the default run converges.

In [ ]:
import subprocess
import shutil
import time

CMD = [
    str(VENV_PY), '-m', 'masoi_training.train_bc',
    '--data', str(DATA),
    '--out', str(OUT),
    '--epochs', '100',
    '--batch-size', '512',
    '--lr', '1e-3',
    '--hidden', '128',
    '--seed', '12345',
    '--model-id', 'policy-p0',
    '--optimizer', 'adamw', '--weight-decay', '0.01',
    '--scheduler', 'cosine', '--warmup-epochs', '1',
    '--grad-clip', '1.0', '--patience', '5',
    '--init', 'orthogonal',
    '--activation', 'silu',
    '--norm', 'layernorm',
    '--value-trunk', 'separate', '--value-weight', '0.5',
]
 
shutil.rmtree(OUT, ignore_errors=True)
start = time.time()
done = subprocess.run(CMD, cwd=str(TRAIN_DIR), env=VENV_ENV)
print(f'\ntotal {time.time() - start:.0f}s  | exit {done.returncode}')
if done.returncode != 0:
    raise SystemExit(f'train_bc failed (exit {done.returncode}). Read the log above.')

## 4. Read the results

The deciding number is **`metrics.test.agreementTieAware`** — tied-for-best vs the
teacher on unseen games. Not `agreement` (penalizes tied moves) and not loss (§17).
Dataset ceiling: **0.984**.

In [ ]:
import json
import pathlib

path = OUT / "metrics.json"
if not path.exists():
    raise SystemExit(
        f"no {path} - training cell (section 3) did NOT finish or failed.\n"
        "Go back to the training cell, read the final 'exit ...' line. Don't edit this cell."
    )
report = json.loads(path.read_text())
m = report["metrics"]

print(f"{'':<11} {'tieAware':>9} {'agreement':>10} {'top-2':>8}")
for name in ("train", "validation", "test"):
    e = m[name]
    print(f"{name:<11} {e.get('agreementTieAware'):>9} {e.get('agreement'):>10} {e.get('top2Agreement'):>8}")

# Diagnose on the SAME metric used for conclusions. Mixing an `agreement` gap with
# `agreementTieAware` conclusions mixes two scales.
gap = m["train"]["agreementTieAware"] - m["validation"]["agreementTieAware"]
print(f"\ntrain - val = {gap:+.4f}  ->", "OVERFIT" if gap > 0.05 else "no overfit")
print("best epoch:", report["bestEpoch"], "/", report["trainingConfig"]["epochs"])

print("\nBy decision type (worst first):")
for kind, score in sorted(m["test"].get("agreementByDecision", {}).items(), key=lambda kv: kv[1]):
    print(f"  {kind:<13} {score}")

print("\nBy role (worst first):")
for role, score in sorted(m["test"].get("agreementByRole", {}).items(), key=lambda kv: kv[1]):
    print(f"  {role:<18} {score}")

print("\nCalibration (ECE - lower is better, > 0.1 is overconfident):")
print(f"  test ece: {m['test'].get('ece')}")
for kind, score in sorted(m["test"].get("eceByDecision", {}).items(), key=lambda kv: kv[1]):
    print(f"  {kind:<13} {score}")

In [ ]:
# Plot via the venv (the kernel may lack matplotlib): render PNG, then display.
import subprocess
import tempfile
from pathlib import Path as _P

PLOT_SRC = '''
import json, sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
out = sys.argv[1]
report = json.load(open(out + '/metrics.json', encoding='utf8'))
history = report['history']
epochs = [row['epoch'] for row in history]
figure, left = plt.subplots(figsize=(8, 4))
left.plot(epochs, [row['trainLoss'] for row in history], label='train loss')
left.set_xlabel('epoch')
left.set_ylabel('loss')
right = left.twinx()
right.plot(
    epochs,
    [r['val_agreementTieAware'] if r.get('val_agreementTieAware') is not None else r.get('val_agreement') for r in history],
    color='tab:orange',
    label='val agreement',
)
right.set_ylabel('agreement')
figure.legend(loc='upper right')
plt.title('Behavior cloning: falling loss is NOT enough, agreement must rise')
plt.savefig(out + '/history.png', dpi=100)
print('plotted', out + '/history.png')
'''

with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False, encoding='utf8') as f:
    f.write(PLOT_SRC)
    _script = f.name
_done = subprocess.run([str(VENV_PY), _script, str(OUT)], cwd=str(TRAIN_DIR), env=VENV_ENV)
_P(_script).unlink()
if _done.returncode != 0:
    raise SystemExit(f'plot failed (exit {_done.returncode})')
from IPython.display import Image, display

display(Image(str(OUT / 'history.png')))